#### Connection to ADLS

In [ ]:
configs = {"fs.azure.account.auth.type": "OAuth",
"fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
"fs.azure.account.oauth2.client.id": "your_client_id",
"fs.azure.account.oauth2.client.secret": 'your_secret_key',
"fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/your_tenant_id/oauth2/token"}


dbutils.fs.mount(
source = "abfss://tokyo-olympic-data@tokioolympicsdata.dfs.core.windows.net", # contrainer@storageacc
mount_point = "/mnt/tokyoolympic",
extra_configs = configs)

#### Storage Mount

In [ ]:
%fs
ls "/mnt/tokyoolympic"

path,name,size,modificationTime
dbfs:/mnt/tokyoolympic/raw-data/,raw-data/,0,1718970670000
dbfs:/mnt/tokyoolympic/transformed-data/,transformed-data/,0,1718797382000


In [ ]:
spark

In [ ]:
athletes = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/tokyoolympic/raw-data/athletes.csv")
coaches = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/tokyoolympic/raw-data/coaches.csv")
entriesgender = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/tokyoolympic/raw-data/gender.csv")
medals = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/tokyoolympic/raw-data/medals.csv")
teams = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/mnt/tokyoolympic/raw-data/teams.csv")
     

In [ ]:
teams.show(5)

+--------+--------------+--------------------+-----+
|TeamName|    Discipline|             Country|Event|
+--------+--------------+--------------------+-----+
| Belgium|3x3 Basketball|             Belgium|  Men|
|   China|3x3 Basketball|People's Republic...|  Men|
|   China|3x3 Basketball|People's Republic...|Women|
|  France|3x3 Basketball|              France|Women|
|   Italy|3x3 Basketball|               Italy|Women|
+--------+--------------+--------------------+-----+
only showing top 5 rows



In [ ]:
dataset_lst = [athletes,coaches,entriesgender,medals,teams]
def show_data_schema(dataset):
    print(f'--------------------------------------------------')
    dataset.show(5)
    dataset.printSchema()

In [ ]:
for dataset in dataset_lst:
    show_data_schema(dataset)

--------------------------------------------------
+-----------------+-------+-------------------+
|       PersonName|Country|         Discipline|
+-----------------+-------+-------------------+
|  AALERUD Katrine| Norway|       Cycling Road|
|      ABAD Nestor|  Spain|Artistic Gymnastics|
|ABAGNALE Giovanni|  Italy|             Rowing|
|   ABALDE Alberto|  Spain|         Basketball|
|    ABALDE Tamara|  Spain|         Basketball|
+-----------------+-------+-------------------+
only showing top 5 rows

root
 |-- PersonName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Discipline: string (nullable = true)

--------------------------------------------------
+---------------+-------------+----------+-----+
|           Name|      Country|Discipline|Event|
+---------------+-------------+----------+-----+
|ABDELMAGID Wael|        Egypt|  Football| NULL|
|      ABE Junya|        Japan|Volleyball| NULL|
|  ABE Katsuhiko|        Japan|Basketball| NULL|
|   ADAMA Cherif|C

#### Top 10 Participating Countries

In [ ]:
athletes.groupBy('Country').count().orderBy("count", ascending=False).limit(10).show()

+--------------------+-----+
|             Country|count|
+--------------------+-----+
|United States of ...|  615|
|               Japan|  586|
|           Australia|  470|
|People's Republic...|  401|
|             Germany|  400|
|              France|  377|
|              Canada|  368|
|       Great Britain|  366|
|               Italy|  356|
|               Spain|  324|
+--------------------+-----+



#### Total number of records with Null event

In [ ]:
coaches.where("Event is null").show(10)
print('No. of records with Null event is: ',coaches.where("Event is null").count())

+--------------------+-------------+----------+-----+
|                Name|      Country|Discipline|Event|
+--------------------+-------------+----------+-----+
|     ABDELMAGID Wael|        Egypt|  Football| NULL|
|           ABE Junya|        Japan|Volleyball| NULL|
|       ABE Katsuhiko|        Japan|Basketball| NULL|
|        ADAMA Cherif|C�te d'Ivoire|  Football| NULL|
|          AGEBA Yuya|        Japan|Volleyball| NULL|
|ALLER CARBALLO Ma...|        Spain|Basketball| NULL|
|           ALY Kamal|        Egypt|  Football| NULL|
| AMAYA GAITAN Fabian|  Puerto Rico|Basketball| NULL|
|    AMO AGUADO Pablo|        Spain|  Football| NULL|
|       BACIU Horatiu|      Romania|  Football| NULL|
+--------------------+-------------+----------+-----+
only showing top 10 rows

No. of records with Null event is:  145


#### Number of participants by Team

In [ ]:
teams_complete = athletes.join(teams, on='Discipline', how='inner')

In [ ]:
teams_complete.groupBy('TeamName').count().orderBy("count", ascending=False).show()

+-----------------+-----+
|         TeamName|count|
+-----------------+-----+
|    United States|22307|
|          Germany|20341|
|            Japan|19731|
|            Italy|19331|
|    Great Britain|18727|
|           France|17210|
|      Netherlands|16832|
|           Poland|14433|
|           Brazil|14045|
|           Canada|13117|
|            China|13111|
|        Australia|13108|
|          Jamaica|10340|
|              ROC| 9901|
|     South Africa| 8026|
|          Belgium| 7661|
|            Spain| 6980|
|          Denmark| 6589|
|      Switzerland| 6580|
|Trinidad & Tobago| 6204|
+-----------------+-----+
only showing top 20 rows



#### Teams who only won Bronze medal

In [ ]:
medals.join(teams, medals.Team_Country==teams.Country, how='inner').where('Gold==0 and Silver==0')\
    .select(['Country','TeamName','Gold','Silver','Bronze','Rank by Total']).show()

+-------------------+---------------+----+------+------+-------------+
|            Country|       TeamName|Gold|Silver|Bronze|Rank by Total|
+-------------------+---------------+----+------+------+-------------+
|         Kazakhstan|     Kazakhstan|   0|     0|     8|           29|
|             Mexico|         Mexico|   0|     0|     4|           47|
|             Mexico|         Mexico|   0|     0|     4|           47|
|Republic of Moldova|Rep. of Moldova|   0|     0|     1|           77|
|         Kazakhstan|     Kazakhstan|   0|     0|     8|           29|
|             Mexico|         Mexico|   0|     0|     4|           47|
|           Botswana|       Botswana|   0|     0|     1|           77|
|              Ghana|          Ghana|   0|     0|     1|           77|
|             Mexico|         Mexico|   0|     0|     4|           47|
|             Mexico|         Mexico|   0|     0|     4|           47|
|             Mexico|  Gaxiola/Rubio|   0|     0|     4|           47|
|     

#### Top countries with the highest number of gold medals

In [ ]:
top_gold_medal_countries = medals.orderBy("Gold", ascending=False)\
    .orderBy("Silver", ascending=False).orderBy("Bronze", ascending=False)\
        .orderBy("Total", ascending=False).show()

+----+--------------------+----+------+------+-----+-------------+
|Rank|        Team_Country|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------------------+----+------+------+-----+-------------+
|   1|United States of ...|  39|    41|    33|  113|            1|
|   2|People's Republic...|  38|    32|    18|   88|            2|
|   5|                 ROC|  20|    28|    23|   71|            3|
|   4|       Great Britain|  22|    21|    22|   65|            4|
|   3|               Japan|  27|    14|    17|   58|            5|
|   6|           Australia|  17|     7|    22|   46|            6|
|  10|               Italy|  10|    10|    20|   40|            7|
|   9|             Germany|  10|    11|    16|   37|            8|
|   7|         Netherlands|  10|    12|    14|   36|            9|
|   8|              France|  10|    12|    11|   33|           10|
|  11|              Canada|   7|     6|    11|   24|           11|
|  12|              Brazil|   7|     6|     8|   21|          

#### Writing data to ADLS

In [ ]:
#writing transformed data to ADLS
athletes.repartition(1).write.mode("overwrite").option("header",'true').csv("/mnt/tokyoolympic/transformed-data/athletes")
entriesgender.repartition(1).write.mode("overwrite").option("header",'true').csv("/mnt/tokyoolympic/transformed-data/entriesgender")
coaches.repartition(1).write.mode("overwrite").option("header",'true').csv("/mnt/tokyoolympic/transformed-data/coaches")
medals.repartition(1).write.mode("overwrite").option("header",'true').csv("/mnt/tokyoolympic/transformed-data/medals")
teams.repartition(1).write.mode("overwrite").option("header",'true').csv("/mnt/tokyoolympic/transformed-data/teams")